In [4]:
!pip install -q aerosandbox robotframework

In [ ]:
import aerosandbox.geometry as ag

# Define parameters for the biplane wings
span = 10.0       # Total wingspan for each wing (symmetric)
chord = 1.0       # Chord length for the rectangular wings
separation = 2.0  # Vertical separation between the lower and upper wing (z-axis)
stagger = 0.5     # Horizontal stagger (x-axis offset) of the upper wing relative to the lower wing
airfoil = ag.Airfoil("naca0012") # Define a simple airfoil shape

# --- Create the Lower Wing ---
# Define the cross-sections for the lower wing.
# For a rectangular wing, we need two cross-sections: root and tip.
# xyz_le is the [x, y, z] coordinate of the leading edge.
lower_wing_xsecs = [
    ag.WingXSec(
        xyz_le=[0, 0, 0],      # Root at the origin [x, y, z]
        chord=chord,
        airfoil=airfoil
    ),
    ag.WingXSec(
        xyz_le=[0, span / 2, 0], # Tip at half span in y, same x and z as root
        chord=chord,
        airfoil=airfoil
    )
]

# Create the Wing object for the lower wing.
# symmetric=True means the defined half-span is mirrored across the XZ plane (y=0).
lower_wing = ag.Wing(
    name="Lower Wing",
    xsecs=lower_wing_xsecs,
    symmetric=True
)

# --- Create the Upper Wing ---
# Define the cross-sections for the upper wing.
# These are similar to the lower wing but offset in x (stagger) and z (separation).
upper_wing_xsecs = [
    ag.WingXSec(
        xyz_le=[stagger, 0, separation],      # Root offset by stagger in x and separation in z
        chord=chord,
        airfoil=airfoil
    ),
    ag.WingXSec(
        xyz_le=[stagger, span / 2, separation], # Tip offset by stagger in x and separation in z, half span in y
        chord=chord,
        airfoil=airfoil
    )
]

# Create the Wing object for the upper wing, also symmetric.
upper_wing = ag.Wing(
    name="Upper Wing",
    xsecs=upper_wing_xsecs,
    symmetric=True
)

# --- Create the Airplane ---
# Create the Airplane object, which is a container for the wings (and other components).
biplane = ag.Airplane(
    name="Simple Biplane",
    wings=[lower_wing, upper_wing] # Add both created wings to the airplane
)

In [4]:
biplane

Airplane 'Simple Biplane' (2 wings, 0 fuselages)

In [5]:
import os
from robot.libdoc import LibraryDocumentation

# Define a placeholder path for a resource file.
# In a real application, this path should point to your actual .robot resource file.
# For this example, we'll assume a file exists at this location.
# You might need to create a dummy file for the code to run without error.
# Example dummy resource file content (e.g., in 'my_resource.robot'):
# *** Keywords ***
# My First Keyword
#     Log    Hello from first keyword
#
# Another Keyword
#     Log    Hello from another keyword
#
resource_file_path = r'C:\Users\preet\Documents\gurveer_stuff\kmit-major-project\CodeSage\my_keywords.resource'

# --- Create a dummy resource file for demonstration if it doesn't exist ---
# This part is just to make the example runnable.
# In a real scenario, you would use your existing resource file.
if not os.path.exists(resource_file_path):
    print(f"Creating a dummy resource file: {resource_file_path}")
    dummy_content = """
*** Keywords ***
My First Keyword
    Log    This is the first keyword

Another Keyword With Args
    [Arguments]    ${arg1}    ${arg2}
    Log    Args: ${arg1}, ${arg2}

Keyword Without Steps
    [Documentation]    This keyword has no steps.
"""
    with open(resource_file_path, 'w') as f:
        f.write(dummy_content.strip())
    print("Dummy file created.")
# --- End of dummy file creation ---

try:
    # Use LibraryDocumentation to parse the resource file
    # The first argument is the path to the resource file
    lib_doc = LibraryDocumentation(resource_file_path)

    print(f"Successfully processed resource file: {resource_file_path}")
    print(f"Resource Name (from file): {lib_doc.name}") # LibraryDoc also has a name attribute

    # Access the keywords attribute which is a list of keyword objects
    keywords = lib_doc.keywords

    if keywords:
        print("\nKeywords found:")
        # Iterate through the keyword objects and print their names
        for kw in keywords:
            print(f"- {kw.name}")
    else:
        print("\nNo keywords found in the resource file.")

except Exception as e:
    print(f"An error occurred while processing the resource file: {e}")
    print(f"Please ensure '{resource_file_path}' is a valid Robot Framework resource file.")

finally:
    # --- Clean up the dummy file ---
    # This part is just for the example's cleanup.
    if os.path.exists(resource_file_path) and "Creating a dummy resource file" in locals().get('dummy_content', ''):
         print(f"\nCleaning up dummy resource file: {resource_file_path}")
         os.remove(resource_file_path)
    # --- End of cleanup ---


Successfully processed resource file: C:\Users\preet\Documents\gurveer_stuff\kmit-major-project\CodeSage\my_keywords.resource
Resource Name (from file): my_keywords

Keywords found:
- Calculate Sum
- Log Message
- Verify Text Contains


In [17]:
import aerosandbox as asb
import aerosandbox.numpy as np

# --- 1. Define the geometry of the monoplane ---

# Define the airfoil for the wing
# Using a standard NACA0012 airfoil
airfoil = asb.Airfoil(name="NACA0012")

# Define the wing cross-sections (root and tip)
# Assuming WingXSec takes xyz position, chord, and airfoil
# Define the right half of the wing and set symmetric=True later
root_xsec = asb.geometry.wing.WingXSec(
    xyz=np.array([0, 0, 0]),  # Root at the origin
    chord=1.0,                # Root chord length (meters)
    airfoil=airfoil
)

tip_xsec = asb.geometry.wing.WingXSec(
    xyz=np.array([0, 5, 0]),  # Tip at y=5m (half-span)
    chord=0.8,                # Tip chord length (meters) - slightly tapered
    airfoil=airfoil
)

# Define the wing using the cross-sections
# Set symmetric=True to model the full wing from the right half
wing = asb.geometry.wing.Wing(
    name="Main Wing",
    xsecs=[root_xsec, tip_xsec],
    symmetric=True
)

# Define the airplane using the wing
# The airplane will automatically calculate reference area, span, and chord
airplane = asb.geometry.airplane.Airplane(
    name="Simple Monoplane",
    wings=[wing]
    # Reference values s_ref, c_ref, b_ref are optional and can be calculated by the Airplane
)

# Print some basic airplane properties (calculated automatically)
print(f"Airplane Reference Area (s_ref): {airplane.s_ref:.3f} m^2")
print(f"Airplane Reference Span (b_ref): {airplane.b_ref:.3f} m")
print(f"Airplane Reference Chord (c_ref): {airplane.c_ref:.3f} m")
print("-" * 20)
# --- 2. Assume a flight condition ---

# Define the operating point (flight condition)
# Assuming OperatingPoint takes velocity, alpha, and density
# Using standard sea level density (1.225 kg/m^3)
flight_velocity = 50  # Airspeed (m/s)
angle_of_attack = 5   # Angle of attack (degrees)
air_density = 1.225   # Air density (kg/m^3) - Standard Sea Level

op_point = asb.performance.operating_point.OperatingPoint(
    velocity=flight_velocity,
    alpha=angle_of_attack,
    density=air_density
)

print(f"Flight Condition:")
print(f"  Velocity: {op_point.velocity} m/s")
print(f"  Alpha: {op_point.alpha} deg")
print(f"  Density: {op_point.density} kg/m^3")
print("-" * 20)
# --- 3. Use LiftingLine to perform aerodynamic analysis ---

# Create a LiftingLine analysis object
# It takes the airplane and the operating point
aero_analysis = asb.aerodynamics.aero_3D.lifting_line.LiftingLine(
    airplane=airplane,
    op_point=op_point
)

# Run the analysis
# The run() method computes forces and moments
print("Running aerodynamic analysis...")
analysis_outputs = aero_analysis.run()
print("Analysis complete.")
print("-" * 20)
# --- 4. Calculate and print the estimated drag force ---

# The analysis outputs are expected to contain aerodynamic coefficients
# Assuming the output contains the coefficient of drag (CD)
cd = analysis_outputs["CD"]

# Calculate dynamic pressure
dynamic_pressure = 0.5 * op_point.density * op_point.velocity**2
# Calculate drag force
# Drag Force = Dynamic Pressure * Reference Area * CD
drag_force = dynamic_pressure * airplane.s_ref * cd
print(f"Estimated Coefficient of Drag (CD): {cd:.4f}")
print(f"Estimated Drag Force: {drag_force:.2f} N")

Airplane Reference Area (s_ref): nan m^2
Airplane Reference Span (b_ref): nan m
Airplane Reference Chord (c_ref): nan m
--------------------


c:\Users\preet\Documents\gurveer_stuff\kmit-major-project\.venv\Lib\site-packages\aerosandbox\geometry\wing.py:1338: RuntimeWarning: invalid value encountered in divide
  return np.array([0, vector[1], vector[2]]) / YZ_magnitude


TypeError: OperatingPoint.__init__() got an unexpected keyword argument 'density'